# CPT Qwen2.5-3B LoRA on SID-v2

Continual pretraining stage for PLUM-style SID-v2 recommendation with a LoRA adapter and a merged CPT checkpoint as output.

Backbone: `Qwen/Qwen2.5-3B` base model.  
Goal: teach the model the relation between Semantic IDs, movie metadata, short descriptions, and train-only user behavior before SFT.

Default profile is intended for a constrained local training setup:

- LoRA instead of full fine-tuning;
- train only LoRA weights plus new SID/schema token rows;
- `max_seq_length=384`;
- micro-batch `2`, gradient accumulation `32`, effective batch `64`;
- if OOM: set micro-batch `1`, accumulation `64`.

## 1. Imports

Run the commented install line once if `peft` is missing in this kernel.

In [1]:
%pip install -U "transformers>=4.57,<5" "peft>=0.19" "accelerate>=1.13"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# %pip install -U "transformers>=4.57,<5" "peft>=0.19" "accelerate>=1.13"
# After installing/updating packages, restart the kernel before continuing.

from pathlib import Path
from importlib import metadata
from packaging.version import Version
import inspect
import json
import os
import random

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")


def require_version(pkg, min_version, install_hint):
    try:
        version = Version(metadata.version(pkg))
    except metadata.PackageNotFoundError as exc:
        raise ImportError(f"{pkg} is missing. Run: {install_hint}") from exc
    if version < Version(min_version):
        raise RuntimeError(
            f"{pkg}=={version} is too old for this notebook.\n"
            f"Run: {install_hint}\n"
            "Then restart the kernel."
        )
    return version

install_hint = '%pip install -U "transformers>=4.57,<5" "peft>=0.19" "accelerate>=1.13"'
TRANSFORMERS_VERSION = require_version("transformers", "4.57.0", install_hint)
PEFT_VERSION = require_version("peft", "0.19.0", install_hint)
ACCELERATE_VERSION = require_version("accelerate", "1.13.0", install_hint)
print("versions:", {"transformers": str(TRANSFORMERS_VERSION), "peft": str(PEFT_VERSION), "accelerate": str(ACCELERATE_VERSION)})

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

versions: {'transformers': '4.57.6', 'peft': '0.19.1', 'accelerate': '1.13.0'}


## 2. Config

The corpus keeps the previous strengthened CPT logic: all movies participate in metadata grounding, descriptions are sharded, and behavior comes only from train split.

In [3]:
ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

BASE_MODEL = "Qwen/Qwen2.5-3B"
RUN_NAME = "cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1"

TRAIN_PATH = ROOT / "data/processed/splits/train.parquet"
USERS_PATH = ROOT / "data/raw/ml-1m/users.dat"
ITEM_PROFILE_PATH = ROOT / "data/processed/item_features/qwen4b_audited_v1_item_profiles.parquet"
SID_ARRAY_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/SIDs_best.npy"
SID_MAPPING_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/sid_mapping_best.parquet"

ARTIFACT_DIR = ROOT / "data/processed/artifacts" / RUN_NAME
MODEL_OUT_DIR = ARTIFACT_DIR / "trainer"
ADAPTER_DIR = ARTIFACT_DIR / "adapter"
FINAL_MODEL_DIR = ARTIFACT_DIR / "final_merged"

SEED = 42
MAX_SEQ_LENGTH = 384
HISTORY_LAST_K = 32
DESC_TEXT_MAX_TOKENS = 48

NUM_SYNTHETIC_EPOCHS = 15
CORE_META_EXAMPLES_PER_ITEM = 2
DESC_SHARDS = 10
BEHAVIOR_RATIO = 0.50
VALIDATION_SIZE = 0.015
MAX_EVAL_EXAMPLES = 512

NUM_TRAIN_EPOCHS = 1.0
DRY_RUN_STEPS = 0
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

PER_DEVICE_BATCH = 2
GRADIENT_ACCUMULATION_STEPS = 32
PER_DEVICE_EVAL_BATCH = 2

EVAL_STEPS = 400
SAVE_STEPS = 400
LOGGING_STEPS = 20
SAVE_TOTAL_LIMIT = 2

INCLUDE_RATINGS = True
INCLUDE_USER_FEATURES = True
SAVE_MERGED_MODEL = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
FP16 = DEVICE == "cuda" and not BF16
PRECISION_MODE = "bf16" if BF16 else ("fp16" if FP16 else "fp32")

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.cuda.empty_cache()

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(DEVICE, PRECISION_MODE)
print("effective batch:", PER_DEVICE_BATCH * GRADIENT_ACCUMULATION_STEPS)

C:\Users\User\plum-ml1m-repro
cuda bf16
effective batch: 64


## 3. Memory profile

This is a practical estimate, not a profiler. For local memory budget, the default should be conservative because only LoRA matrices and new domain-token rows are trainable.

In [4]:
QWEN25_3B_PARAMS = 3.09e9
BASE_WEIGHT_GB = QWEN25_3B_PARAMS * 2 / 1024**3
EFFECTIVE_BATCH = PER_DEVICE_BATCH * GRADIENT_ACCUMULATION_STEPS
EFFECTIVE_TOKENS = EFFECTIVE_BATCH * MAX_SEQ_LENGTH

memory_plan = {
    "base_model": BASE_MODEL,
    "precision": PRECISION_MODE,
    "base_weights_gb_bf16_or_fp16_est": round(BASE_WEIGHT_GB, 2),
    "max_seq_length": MAX_SEQ_LENGTH,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch": EFFECTIVE_BATCH,
    "effective_tokens_per_update": EFFECTIVE_TOKENS,
    "oom_fallback": {
        "per_device_batch": 1,
        "gradient_accumulation_steps": 64,
        "effective_batch": 64,
    },
    "notes": [
        "Use gradient checkpointing.",
        "Use trainable_token_indices for new SID/schema tokens if PEFT supports it.",
        "If PEFT is old and falls back to full embed/lm_head modules_to_save, use batch=1.",
    ],
}
print(json.dumps(memory_plan, indent=2))

{
  "base_model": "Qwen/Qwen2.5-3B",
  "precision": "bf16",
  "base_weights_gb_bf16_or_fp16_est": 5.76,
  "max_seq_length": 384,
  "per_device_batch": 2,
  "gradient_accumulation_steps": 32,
  "effective_batch": 64,
  "effective_tokens_per_update": 24576,
  "oom_fallback": {
    "per_device_batch": 1,
    "gradient_accumulation_steps": 64,
    "effective_batch": 64
  },
  "notes": [
    "Use gradient checkpointing.",
    "Use trainable_token_indices for new SID/schema tokens if PEFT supports it.",
    "If PEFT is old and falls back to full embed/lm_head modules_to_save, use batch=1."
  ]
}


## 4. Load tables

CPT behavior examples use only `train.parquet`. Validation/test recommendation targets are not used here.

In [5]:
train = pd.read_parquet(TRAIN_PATH)
users = pd.read_csv(
    USERS_PATH,
    sep="::",
    engine="python",
    names=["user_id", "gender", "age", "occupation", "zip"],
    encoding="latin-1",
)
profiles = pd.read_parquet(ITEM_PROFILE_PATH).sort_values("item_idx").reset_index(drop=True)
sid_mapping = pd.read_parquet(SID_MAPPING_PATH).sort_values("item_idx").reset_index(drop=True)
sids = np.load(SID_ARRAY_PATH)

assert train[["user_id", "user_idx", "item_idx", "rating", "timestamp"]].notna().all().all()
assert np.array_equal(profiles["item_idx"].to_numpy(), np.arange(len(profiles)))
assert np.array_equal(sid_mapping["item_idx"].to_numpy(), np.arange(len(sid_mapping)))
assert sids.shape == (len(profiles), 4)

profiles["title_clean"] = profiles["title_clean"].fillna(profiles["title"].astype(str))
profiles["year_text"] = profiles["year_text"].fillna(profiles["year"].astype(str))
profiles["genres_text"] = profiles["genres_text"].fillna(profiles["genres"].astype(str).str.replace("|", ", ", regex=False))
profiles["description_text"] = profiles["description_text"].fillna("")

print("train rows:", len(train))
print("users:", len(users))
print("items:", len(profiles))
print("sids:", sids.shape)
profiles[["item_idx", "movie_id", "title", "year_text", "genres_text", "description_text"]].head()

train rows: 988129
users: 6040
items: 3706
sids: (3706, 4)


,item_idx,movie_id,title,year_text,genres_text,description_text
0,0,1,Toy Story (1995),1995,"Animation, Children's, Comedy","Plot overview: Led by Woody, Andy's toys live ..."
1,1,2,Jumanji (1995),1995,"Adventure, Children's, Fantasy",Plot overview: When siblings Judy and Peter di...
2,2,3,Grumpier Old Men (1995),1995,"Comedy, Romance",Plot overview: A family wedding reignites the ...
3,3,4,Waiting to Exhale (1995),1995,"Comedy, Drama","Plot overview: Cheated on, mistreated and step..."
4,4,5,Father of the Bride Part II (1995),1995,Comedy,Plot overview: Just when George Banks has reco...


## 5. Tokenizer and SID schema

New SID/schema tokens are added explicitly. The notebook tries to train only these new token rows with PEFT `trainable_token_indices`; this is the main memory-saving detail.

In [6]:
BOS = "<bos>"
EOS = "<eos>"
PAD = "<pad>"
UNK = "<unk>"
USER_OPEN = "<user>"
USER_CLOSE = "</user>"
HIST = "<hist>"
EVENT_OPEN = "<e>"
EVENT_CLOSE = "</e>"
META_OPEN = "<meta>"
META_CLOSE = "</meta>"
DESC_OPEN = "<desc>"
DESC_CLOSE = "</desc>"
TASK_META = "<task_meta>"
TASK_SID = "<task_sid>"
TASK_DESC = "<task_desc>"
TASK_HIST = "<task_hist>"

SCHEMA_TOKENS = [
    USER_OPEN, USER_CLOSE, HIST, EVENT_OPEN, EVENT_CLOSE,
    META_OPEN, META_CLOSE, DESC_OPEN, DESC_CLOSE,
    TASK_META, TASK_SID, TASK_DESC, TASK_HIST,
]


def sid_tokens(sid):
    return [f"<sid_{level}_{int(code)}>" for level, code in enumerate(sid)]


def sid_text(item_idx):
    return " ".join(sid_tokens(sids[int(item_idx)]))


def rating_token(rating):
    return f"<rat_{int(rating)}>"


def user_tokens(row):
    return [f"<gen_{row.gender}>", f"<age_{int(row.age)}>", f"<occ_{int(row.occupation)}>"]


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True, trust_remote_code=False)
base_vocab_size = len(tokenizer)

special_map = {
    "bos_token": BOS,
    "eos_token": EOS,
    "pad_token": PAD,
    "unk_token": UNK,
}
tokenizer.add_special_tokens(special_map)

extra_tokens = set(SCHEMA_TOKENS)
extra_tokens.update(rating_token(r) for r in range(1, 6))
extra_tokens.update(token for sid in sids for token in sid_tokens(sid))

if INCLUDE_USER_FEATURES:
    extra_tokens.update(f"<gen_{v}>" for v in users["gender"].dropna().unique())
    extra_tokens.update(f"<age_{int(v)}>" for v in users["age"].dropna().unique())
    extra_tokens.update(f"<occ_{int(v)}>" for v in users["occupation"].dropna().unique())

n_added = tokenizer.add_tokens(sorted(extra_tokens), special_tokens=True)
DOMAIN_TOKENS = [BOS, EOS, PAD, UNK, *sorted(extra_tokens)]
DOMAIN_TOKEN_IDS = sorted(set(int(x) for x in tokenizer.convert_tokens_to_ids(DOMAIN_TOKENS)))

assert all(x >= 0 for x in DOMAIN_TOKEN_IDS)
print("base vocab:", base_vocab_size)
print("new tokenizer length:", len(tokenizer))
print("added tokens:", len(tokenizer) - base_vocab_size)
print("domain trainable token ids:", len(DOMAIN_TOKEN_IDS))

base vocab: 151665
new tokenizer length: 152049
added tokens: 384
domain trainable token ids: 385


## 6. Metadata and description templates

The templates intentionally mix SID-to-metadata and metadata-to-SID directions.

In [7]:
CORE_TEMPLATES = [
    "{bos} {task} {meta_open} Movie {sid} has title: {title}. Release year: {year}. Genres: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Semantic ID {sid} refers to movie: {title}. Year: {year}. Genres: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Item {sid}. Title: {title}. Year: {year}. Genre labels: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} For {sid}, the movie metadata is title {title}, year {year}, genres {genres}. {meta_close} {eos}",
]

REVERSE_TEMPLATES = [
    "{bos} {task} {meta_open} Movie title: {title}. Release year: {year}. Genres: {genres}. Semantic ID: {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} The movie {title} from {year}, with genres {genres}, maps to semantic ID {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Genres: {genres}. Title: {title}. Year: {year}. SID: {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Identify SID for movie {title} ({year}), genres {genres}: {sid}. {meta_close} {eos}",
]

FIELD_TEMPLATES = [
    "{bos} {task} {meta_open} Title of {sid}: {title}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Genres of {sid}: {genres}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Release year of {sid}: {year}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Movie {title} has semantic ID {sid}. {meta_close} {eos}",
    "{bos} {task} {meta_open} Movie with genres {genres} and title {title} uses SID {sid}. {meta_close} {eos}",
]

DESC_TEMPLATES = [
    "{bos} {task} {desc_open} Movie {sid}. Title: {title}. Short plot: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Plot overview for {sid}, {title}: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Title: {title}. Genres: {genres}. Semantic ID: {sid}. Plot: {desc} {desc_close} {eos}",
    "{bos} {task} {desc_open} Plot: {desc} Semantic ID: {sid}. Movie title: {title}. {desc_close} {eos}",
]


def encode_text(text, max_length=MAX_SEQ_LENGTH):
    return tokenizer.encode(text, add_special_tokens=False, truncation=True, max_length=max_length)


def shorten_text(text, max_tokens=DESC_TEXT_MAX_TOKENS):
    ids = tokenizer.encode(str(text), add_special_tokens=False)
    return tokenizer.decode(ids[:max_tokens]).strip()


def fmt(template, row, task):
    return template.format(
        bos=BOS,
        eos=EOS,
        task=task,
        meta_open=META_OPEN,
        meta_close=META_CLOSE,
        desc_open=DESC_OPEN,
        desc_close=DESC_CLOSE,
        sid=sid_text(row.item_idx),
        title=str(row.title_clean).strip(),
        year=str(row.year_text).strip(),
        genres=str(row.genres_text).strip(),
        desc=shorten_text(row.description_text),
    )


def has_description(row):
    text = str(row.description_text).strip()
    return bool(text) and text.lower() != "nan"

## 7. Behavior examples

Behavior windows are train-only. This keeps CPT from leaking validation/test recommendation targets.

In [8]:
users_by_id = users.set_index("user_id", drop=False)
train_sorted = train.sort_values(["user_idx", "timestamp", "pos", "item_idx"], kind="mergesort").reset_index(drop=True)
user_events = {
    int(user_id): group[["item_idx", "rating", "timestamp"]].to_dict("records")
    for user_id, group in train_sorted.groupby("user_id", sort=False)
}
user_ids = np.array(list(user_events.keys()))


def behavior_event(event):
    tokens = [EVENT_OPEN]
    tokens.extend(sid_tokens(sids[int(event["item_idx"])]))
    if INCLUDE_RATINGS:
        tokens.append(rating_token(event["rating"]))
    tokens.append(EVENT_CLOSE)
    return tokens


def fit_behavior_tokens(prefix, events):
    selected = list(events[-HISTORY_LAST_K:])
    while selected:
        tokens = prefix + [tok for event in selected for tok in event] + [EOS]
        ids = tokenizer.convert_tokens_to_ids(tokens)
        if len(ids) <= MAX_SEQ_LENGTH:
            return ids
        selected = selected[1:]
    return tokenizer.convert_tokens_to_ids(prefix + [EOS])


def make_behavior_example(user_id, rng):
    events = user_events[int(user_id)]
    cutoff = len(events) if len(events) <= 1 else int(rng.integers(1, len(events) + 1))
    window = events[:cutoff]

    prefix = [BOS, TASK_HIST]
    if INCLUDE_USER_FEATURES and int(user_id) in users_by_id.index:
        prefix.extend([USER_OPEN, *user_tokens(users_by_id.loc[int(user_id)]), USER_CLOSE])
    prefix.append(HIST)

    return fit_behavior_tokens(prefix, [behavior_event(event) for event in window])

## 8. Build curriculum corpus

`NUM_SYNTHETIC_EPOCHS=15` is the default lower edge of the strengthened 15-20 epoch CPT setup. It keeps all movies in every metadata pass while staying realistic for the first 3B run on the local memory budget.

In [9]:
rows = []
epoch_stats = []
profile_rows = list(profiles.itertuples(index=False))

for epoch in range(NUM_SYNTHETIC_EPOCHS):
    rng = np.random.default_rng(SEED + epoch)
    epoch_rows = []

    for row in profile_rows:
        template_plan = [CORE_TEMPLATES[int(rng.integers(0, len(CORE_TEMPLATES)))]]
        extra_pool = REVERSE_TEMPLATES + FIELD_TEMPLATES
        while len(template_plan) < CORE_META_EXAMPLES_PER_ITEM:
            template_plan.append(extra_pool[int(rng.integers(0, len(extra_pool)))])

        for j, template in enumerate(template_plan):
            task = TASK_META if j == 0 else TASK_SID
            source = "metadata_core" if j == 0 else "metadata_reverse_or_field"
            epoch_rows.append({
                "input_ids": encode_text(fmt(template, row, task)),
                "source": source,
                "synthetic_epoch": epoch,
                "item_idx": int(row.item_idx),
            })

    desc_candidates = [
        row for row in profile_rows
        if has_description(row) and int(row.item_idx) % DESC_SHARDS == epoch % DESC_SHARDS
    ]
    rng.shuffle(desc_candidates)
    for row in desc_candidates:
        desc_template = DESC_TEMPLATES[int(rng.integers(0, len(DESC_TEMPLATES)))]
        epoch_rows.append({
            "input_ids": encode_text(fmt(desc_template, row, TASK_DESC)),
            "source": "metadata_description",
            "synthetic_epoch": epoch,
            "item_idx": int(row.item_idx),
        })

    n_metadata = len(epoch_rows)
    n_behavior = int(round(n_metadata * BEHAVIOR_RATIO / (1.0 - BEHAVIOR_RATIO)))

    selected_users = list(user_ids)
    if n_behavior > len(selected_users):
        extra = rng.choice(user_ids, size=n_behavior - len(selected_users), replace=True).tolist()
        selected_users.extend(extra)
    else:
        rng.shuffle(selected_users)
        selected_users = selected_users[:n_behavior]

    for user_id in selected_users:
        epoch_rows.append({
            "input_ids": make_behavior_example(int(user_id), rng),
            "source": "behavior_window",
            "synthetic_epoch": epoch,
            "item_idx": -1,
        })

    rng.shuffle(epoch_rows)
    rows.extend(epoch_rows)
    epoch_stats.append(pd.Series([r["source"] for r in epoch_rows]).value_counts().to_dict())

rng = np.random.default_rng(SEED)
rng.shuffle(rows)

val_n = min(MAX_EVAL_EXAMPLES, max(1, int(len(rows) * VALIDATION_SIZE)))
val_idx = rng.choice(len(rows), size=val_n, replace=False)
val_rows = [rows[int(i)] for i in val_idx]
train_rows = rows

source_counts = pd.Series([r["source"] for r in rows]).value_counts().to_dict()
lengths = np.array([len(r["input_ids"]) for r in rows], dtype=np.int32)

curriculum_stats = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "total_examples": len(rows),
    "train_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "validation_is_monitor_sample_from_curriculum": True,
    "num_synthetic_epochs": NUM_SYNTHETIC_EPOCHS,
    "items": len(profiles),
    "all_items_in_core_metadata_each_epoch": True,
    "core_meta_examples_per_item_per_epoch": CORE_META_EXAMPLES_PER_ITEM,
    "desc_shards": DESC_SHARDS,
    "behavior_ratio_target": BEHAVIOR_RATIO,
    "source_counts": source_counts,
    "length_min": int(lengths.min()),
    "length_p50": int(np.percentile(lengths, 50)),
    "length_p95": int(np.percentile(lengths, 95)),
    "length_max": int(lengths.max()),
    "max_seq_length": MAX_SEQ_LENGTH,
    "history_last_k": HISTORY_LAST_K,
    "desc_text_max_tokens": DESC_TEXT_MAX_TOKENS,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch": EFFECTIVE_BATCH,
}

print(json.dumps(curriculum_stats, indent=2))
pd.DataFrame(epoch_stats).fillna(0).astype(int)

{
  "run_name": "cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1",
  "base_model": "Qwen/Qwen2.5-3B",
  "total_examples": 233482,
  "train_examples": 233482,
  "validation_examples": 512,
  "validation_is_monitor_sample_from_curriculum": true,
  "num_synthetic_epochs": 15,
  "items": 3706,
  "all_items_in_core_metadata_each_epoch": true,
  "core_meta_examples_per_item_per_epoch": 2,
  "desc_shards": 10,
  "behavior_ratio_target": 0.5,
  "source_counts": {
    "behavior_window": 116741,
    "metadata_core": 55590,
    "metadata_reverse_or_field": 55590,
    "metadata_description": 5561
  },
  "length_min": 16,
  "length_p50": 49,
  "length_p95": 233,
  "length_max": 233,
  "max_seq_length": 384,
  "history_last_k": 32,
  "desc_text_max_tokens": 48,
  "per_device_batch": 2,
  "gradient_accumulation_steps": 32,
  "effective_batch": 64
}


,behavior_window,metadata_reverse_or_field,metadata_core,metadata_description
0,7783,3706,3706,371
1,7783,3706,3706,371
2,7783,3706,3706,371
3,7783,3706,3706,371
4,7783,3706,3706,371
5,7783,3706,3706,371
6,7782,3706,3706,370
7,7782,3706,3706,370
8,7782,3706,3706,370
9,7782,3706,3706,370


## 9. Save curriculum artifacts

The corpus and tokenizer are saved before model loading, so training can be restarted from this point.

In [10]:
tokenizer.save_pretrained(ARTIFACT_DIR / "tokenizer")
np.save(ARTIFACT_DIR / "SIDs_best.npy", sids)

torch.save(
    {
        "train_rows": train_rows,
        "val_rows": val_rows,
        "curriculum_stats": curriculum_stats,
        "domain_tokens": DOMAIN_TOKENS,
        "domain_token_ids": DOMAIN_TOKEN_IDS,
    },
    ARTIFACT_DIR / "cpt_curriculum_corpus.pt",
)

manifest = {
    "run_name": RUN_NAME,
    "base_model": BASE_MODEL,
    "train_path": str(TRAIN_PATH),
    "item_profile_path": str(ITEM_PROFILE_PATH),
    "sid_array_path": str(SID_ARRAY_PATH),
    "sid_mapping_path": str(SID_MAPPING_PATH),
    "uses_recsys_val_test_as_behavior": False,
    "memory_plan": memory_plan,
    "curriculum_stats": curriculum_stats,
}
with (ARTIFACT_DIR / "manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(ARTIFACT_DIR)

C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1


## 10. Load Qwen2.5-3B with LoRA

This cell is the memory-sensitive part. Default target modules are attention + MLP projections. New SID/schema token rows are trainable through `trainable_token_indices` when available.

In [11]:
class LMListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return {"input_ids": torch.tensor(self.rows[idx]["input_ids"], dtype=torch.long)}


payload = torch.load(ARTIFACT_DIR / "cpt_curriculum_corpus.pt", weights_only=False)
train_ds = LMListDataset(payload["train_rows"])
val_ds = LMListDataset(payload["val_rows"])

tokenizer = AutoTokenizer.from_pretrained(ARTIFACT_DIR / "tokenizer", use_fast=True)

dtype = torch.bfloat16 if BF16 else (torch.float16 if FP16 else torch.float32)
model_kwargs = {
    "dtype": dtype,
    "trust_remote_code": False,
}
try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, attn_implementation="sdpa", **model_kwargs)
except TypeError:
    model_kwargs["torch_dtype"] = model_kwargs.pop("dtype")
    try:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, attn_implementation="sdpa", **model_kwargs)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)

model.resize_token_embeddings(len(tokenizer))
model.config.bos_token_id = tokenizer.bos_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_kwargs = {
    "task_type": TaskType.CAUSAL_LM,
    "r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "bias": "none",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}

lora_params = inspect.signature(LoraConfig).parameters
if "trainable_token_indices" in lora_params:
    lora_kwargs["trainable_token_indices"] = {"embed_tokens": payload["domain_token_ids"]}
else:
    lora_kwargs["modules_to_save"] = ["embed_tokens", "lm_head"]
    print("PEFT lacks trainable_token_indices; falling back to full embed/lm_head training. Consider batch=1.")

peft_config = LoraConfig(**lora_kwargs)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.to(DEVICE)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)
print("precision:", PRECISION_MODE, "train examples:", len(train_ds), "val examples:", len(val_ds))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


trainable params: 30,722,048 || all params: 3,116,892,160 || trainable%: 0.9857
precision: bf16 train examples: 233482 val examples: 512


## 11. Trainer

If CUDA OOM happens in the next cell, restart kernel and set `PER_DEVICE_BATCH=1`, `GRADIENT_ACCUMULATION_STEPS=64` in config.

In [12]:
args_kwargs = {
    "output_dir": str(MODEL_OUT_DIR),
    "overwrite_output_dir": True,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": DRY_RUN_STEPS if DRY_RUN_STEPS > 0 else -1,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "per_device_train_batch_size": PER_DEVICE_BATCH,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "gradient_checkpointing": True,
    "logging_steps": LOGGING_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,
    "save_strategy": "steps",
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "fp16": FP16,
    "bf16": BF16,
    "tf32": DEVICE == "cuda",
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "dataloader_num_workers": 0,
    "dataloader_pin_memory": DEVICE == "cuda",
    "remove_unused_columns": False,
    "optim": "adamw_torch_fused" if DEVICE == "cuda" else "adamw_torch",
}

args_params = inspect.signature(TrainingArguments).parameters
if "eval_strategy" in args_params:
    args_kwargs["eval_strategy"] = "steps"
else:
    args_kwargs["evaluation_strategy"] = "steps"

training_args = TrainingArguments(**{k: v for k, v in args_kwargs.items() if k in args_params})
if DRY_RUN_STEPS > 0:
    print(f"DRY RUN MODE: training will stop after {DRY_RUN_STEPS} step(s).")

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_ds,
    "eval_dataset": val_ds,
    "data_collator": collator,
}
trainer_params = inspect.signature(Trainer).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer

## 12. Train CPT

This is the long cell. For a one-step smoke test set `DRY_RUN_STEPS=1` in config, restart kernel, and rerun.

In [13]:
train_output = trainer.train()

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

result = {
    "dry_run": DRY_RUN_STEPS > 0,
    "train_output": train_output.metrics,
    "curriculum_stats": payload["curriculum_stats"],
    "adapter_dir": str(ADAPTER_DIR),
}

if SAVE_MERGED_MODEL and DRY_RUN_STEPS == 0:
    try:
        torch.cuda.empty_cache() if DEVICE == "cuda" else None
        FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
        merged = trainer.model.merge_and_unload()
        merged.save_pretrained(str(FINAL_MODEL_DIR), safe_serialization=True, max_shard_size="2GB")
        tokenizer.save_pretrained(str(FINAL_MODEL_DIR))
        result["final_merged_model_dir"] = str(FINAL_MODEL_DIR)
    except Exception as exc:
        result["merge_error"] = repr(exc)
        result["note"] = "Adapter was saved successfully; merged model export failed. Use adapter_dir for SFT or rerun merge separately."

with (ARTIFACT_DIR / "train_result.json").open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151666, 'bos_token_id': 151665, 'pad_token_id': 151667}.


Step,Training Loss,Validation Loss
400,1.110200,1.043290
800,0.991500,0.927522
1200,0.976200,0.876411
1600,0.945300,0.840285
2000,0.923800,0.816800
2400,0.895300,0.793856
2800,0.867100,0.771573
3200,0.852700,0.748946
3600,0.843200,0.735039


C:\Users\User\.conda\conda.new\Lib\site-packages\peft\utils\save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
C:\Users\User\.conda\conda.new\Lib\site-packages\peft\utils\save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
C:\Users\User\.conda\conda.new\Lib\site-packages\peft\utils\save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
C:\Users\User\.conda\conda.new\Lib\site-packages\peft\utils\save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
C:\Users\User\.conda\conda.new\Lib\site-packages\peft\utils\save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embe

{'dry_run': False,
 'train_output': {'train_runtime': 35471.5769,
  'train_samples_per_second': 6.582,
  'train_steps_per_second': 0.103,
  'total_flos': 6.303782097465508e+17,
  'train_loss': 1.1017206997765552,
  'epoch': 1.0},
 'curriculum_stats': {'run_name': 'cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1',
  'base_model': 'Qwen/Qwen2.5-3B',
  'total_examples': 233482,
  'train_examples': 233482,
  'validation_examples': 512,
  'validation_is_monitor_sample_from_curriculum': True,
  'num_synthetic_epochs': 15,
  'items': 3706,
  'all_items_in_core_metadata_each_epoch': True,
  'core_meta_examples_per_item_per_epoch': 2,
  'desc_shards': 10,
  'behavior_ratio_target': 0.5,
  'source_counts': {'behavior_window': 116741,
   'metadata_core': 55590,
   'metadata_reverse_or_field': 55590,
   'metadata_description': 5561},
  'length_min': 16,
  'length_p50': 49,
  'length_p95': 233,
  'length_max': 233,
  'max_seq_length': 384,
  'history_last_k': 32,
  'desc_text_max_tokens': 48,
  'per_

## 13. Quick sanity generation

These checks are not recommender metrics. They only show whether CPT learned the SID/metadata surface form.

In [14]:
import sys


def safe_for_stdout(text):
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    return str(text).encode(encoding, errors="backslashreplace").decode(encoding, errors="replace")


if FINAL_MODEL_DIR.exists():
    MODEL_DIR = FINAL_MODEL_DIR
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    load_kwargs = {"dtype": torch.bfloat16 if BF16 else (torch.float16 if FP16 else torch.float32), "trust_remote_code": False}
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, **load_kwargs).to(DEVICE)
    except TypeError:
        load_kwargs["torch_dtype"] = load_kwargs.pop("dtype")
        model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, **load_kwargs).to(DEVICE)
else:
    MODEL_DIR = ADAPTER_DIR
    if not MODEL_DIR.exists():
        raise FileNotFoundError("No CPT checkpoint found yet. Run the training cell first, or set MODEL_DIR manually.")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
    load_kwargs = {"dtype": torch.bfloat16 if BF16 else (torch.float16 if FP16 else torch.float32), "trust_remote_code": False}
    try:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)
    except TypeError:
        load_kwargs["torch_dtype"] = load_kwargs.pop("dtype")
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **load_kwargs)
    base.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base, MODEL_DIR).to(DEVICE)

model.eval()


def generate(prompt, max_new_tokens=80, do_sample=False, temperature=0.8, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0], skip_special_tokens=False)

for item_idx in [0, 1, 1104, 1178, 1192]:
    row = profiles.iloc[item_idx]
    prompt = f"{BOS} {TASK_META} {META_OPEN} Movie {sid_text(item_idx)} has title:"
    print("=" * 100)
    print("TRUE:", row["title"], "|", row["year_text"], "|", row["genres_text"])
    print(safe_for_stdout(generate(prompt, max_new_tokens=70, do_sample=False)))

The tokenizer you are loading from 'C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1\final_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

TRUE: Toy Story (1995) | 1995 | Animation, Children's, Comedy
<bos> <task_meta> <meta> Movie <sid_0_32> <sid_1_134> <sid_2_20> <sid_3_18> has title: Toy Story. Release year: 1995. Genres: Animation, Children's, Comedy. </meta> <eos>
TRUE: Jumanji (1995) | 1995 | Adventure, Children's, Fantasy
<bos> <task_meta> <meta> Movie <sid_0_469> <sid_1_18> <sid_2_69> <sid_3_31> has title: Jumanji. Release year: 1995. Genres: Adventure, Children's, Fantasy. </meta> <eos>
TRUE: One Flew Over the Cuckoo's Nest (1975) | 1975 | Drama
<bos> <task_meta> <meta> Movie <sid_0_227> <sid_1_113> <sid_2_59> <sid_3_30> has title: One Flew Over the Cuckoo's Nest. Release year: 1975. Genres: Drama. </meta> <eos>
TRUE: Back to the Future (1985) | 1985 | Comedy, Sci-Fi
<bos> <task_meta> <meta> Movie <sid_0_148> <sid_1_28> <sid_2_66> <sid_3_3> has title: Back to the Future. Release year: 1985. Genres: Comedy, Sci-Fi. </meta> <eos>
TRUE: Big Sleep, The (1946) | 1946 | Film-Noir, Mystery
<bos> <task_meta> <meta> Movie

## 14. Notes for the next SFT stage

Use `final_merged` as the base CPT checkpoint if it exists. If only `adapter` exists, SFT should load `Qwen/Qwen2.5-3B`, resize tokenizer from the adapter directory, then load the CPT PEFT adapter before SFT.

Recommended first SFT profile after this CPT:

- same target-only next-watched objective;
- history window `16` first, then ablate `32` and `64`;
- trie-constrained beam decoding;
- sample validation every epoch, full validation only on best checkpoint.